Load packages and modules

In [ ]:
import glob
import os
import time
import warnings
import sklearn
import shap
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import enlighten
import matplotlib.pyplot as plt
import io


from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from xgboost import XGBClassifier

from ipywidgets import IntProgress
from IPython.display import display

Function for doing Shap values and storing for plotting

In [ ]:
def features_ml_models(X, y, random_seed, variables, sp= None, factor = None,i =None, j=None,  is_binary=False, is_balanced = False, plot = False):
    """Train and evaluate ML models on feature data, extracting both metrics and confusion matrices.

    Args:
        X (pd.DataFrame): Feature matrix
        y (pd.Series): Target variable (either binary or multiclass)
        random_seed (int): Random seed for train/test split
        is_binary (bool): Whether this is binary classification

    Returns:
        tuple: (metrics_df, confusion_df, SHAP_values) with model performance metrics and confusion matrix data
    """
    X = X.reset_index(drop=True)
    
    # Encode target variables
    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y)

    # Special handling for binary classification to ensure proper encoding
    if is_binary and len(target_encoder.classes_) == 2:
        # Ensure "Active" is encoded as 1 and "Inactive" as 0 if these are the class names
        if (
            "Active" in target_encoder.classes_
            and "Inactive" in target_encoder.classes_
        ):
            if target_encoder.transform(["Active"])[0] != 1:
                target_encoder.classes_ = target_encoder.classes_[::-1]
                y_encoded = target_encoder.transform(y)

        # Train and test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, stratify=y_encoded, test_size=0.25, random_state=random_seed
    )

    # Define the models
    models = {
       # "RF": RandomForestClassifier(n_estimators=100, random_state=42),
        "XGB": XGBClassifier(random_state=random_seed),
        #"SVC": SVC(kernel="rbf", cache_size=1000, probability=True, random_state=42),
    }

    # Create lists to store the results and confusion matrix data
    metrics_results = []
    confusion_data = []

    # Iterate over the models
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for model_name, model in models.items():
            start_time = time.time()


            # Fit the model with 
            
                        # Fit the model with
            if is_balanced == False:  
                model.fit(X_train, y_train, sample_weight=compute_sample_weight("balanced", y_train) )
            
            else:
                model.fit(X_train, y_train )

            # Predictions
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)

            # Calculate accuracy
            accuracy = model.score(X_test, y_test)

            # Calculate ROC AUC - handle binary and multiclass differently
            if is_binary:
                roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
            else:
                roc_auc = roc_auc_score(
                    y_test, y_pred_proba, multi_class="ovr", average="weighted"
                )

            # Calculate balanced accuracy
            balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

            # Calculate F1 score
            f1 = f1_score(y_test, y_pred, average="weighted")

            # Calculate precision
            precision = precision_score(
                y_test, y_pred, average="weighted", zero_division=0
            )

            # Calculate recall
            recall = recall_score(y_test, y_pred, average="weighted")

            # SHAP values 

            if model_name == "XGB":
                #print("XGB")
                #explainer = shap.TreeExplainer(model)
                explainer = shap.TreeExplainer(model, model_output= "raw" )
                explanation = explainer(X_train)

                shap_values = explanation.values
                # make sure the SHAP values add up to marginal predictions
                #np.abs(shap_values.sum(axis=1) + explanation.base_values - y_test).max()
                
                #new_index = [ix for ix_test_fold in ix_test for ix in ix_test_fold]
                if explanation is not None: # Ensure you only add valid explanations
                    explanations_list.append(explanation)
                    titles_list.append(f"{sp} - {factor}")
                    
                if plot == True :
                    #
                    shap.plots.beeswarm(explanation, plot_size =None, ax=axes[i * 2 + j]) #, feature_names=X.columns)
                    #axes[i * 2 + j].set_title(f"{sp} - {factor}")
                    #shap.plots.bar(explanation)
                    #shap.plots.scatter(shap_values[:, "Relationship"], color=shap_values)
                
                #shap_values_df = pd.DataFrame(shap_values,  columns=X.columns)
                vals= np.abs(shap_values).mean(0)

                feature_importance = pd.DataFrame(list(zip(X_train.columns, vals)))
                feature_importance_df = feature_importance.T


            train_time = time.time() - start_time

            # Append the metrics to the results list


            # Get and process confusion matrix
            cm = confusion_matrix(y_test, y_pred)

            # Convert confusion matrix to DataFrame format
            for i in range(len(cm)):
                for j in range(len(cm)):
                    confusion_data.append(
                        {
                            "model": model_name,
                            "actual_label": i,
                            "predicted_label": j,
                            "count": cm[i][j],
                        }
                    )

            #print(f"{model_name} Done")

    # Convert results to DataFrames
    confusion_df = pd.DataFrame(confusion_data)

    # Map the encoded labels back to original classes if possible
    original_classes = target_encoder.classes_
    if len(confusion_df) > 0:
        confusion_df["actual_label"] = confusion_df["actual_label"].apply(
            lambda x: original_classes[x] if x < len(original_classes) else x
        )
        confusion_df["predicted_label"] = confusion_df["predicted_label"].apply(
            lambda x: original_classes[x] if x < len(original_classes) else x
        )

        metrics_results.append(
        {
            "model": model_name,
            "roc_auc": roc_auc,
            "accuracy": accuracy,
            "balanced_acc": balanced_accuracy,
            "f1_score": f1,
            "precision": precision,
            "recall": recall,
            "time": train_time,
            "TP" : confusion_df.loc[3, "count"],
            "TN" : confusion_df.loc[0, "count"],
            "FP" : confusion_df.loc[1, "count"],
            "FN" :  confusion_df.loc[2, "count"]
        }

    )
        metrics_df = pd.DataFrame(metrics_results)
        
    return metrics_df, feature_importance_df


Load data and variable names

In [ ]:
# Configuration
txtfiles = []

for file in glob.glob("../../../results/species/AssignVariables/*_allv.csv"):
    txtfiles.append(file)

S_levels = ["S2"] #just doing Beeswarm plots for the S2 period
variables = ["PCA_1", "PCA_2", "HM", "trade", "MPD"]

Run model for each species at S2

In [ ]:
# Initialize storage
metrics_rows_w = []
shap_values_store_w = {}

replicates = 1

manager = enlighten.get_manager()
ticks = manager.counter(total=13, desc='Ticks', unit='ticks')
tocks = manager.counter(total=26, desc='Tocks', unit='tocks')

explanations_list = []
titles_list = []

# Loop structure
for i, file in enumerate(txtfiles):

    a = pd.read_csv(file)
    seed = 0 
    print(a.loc[1, 'Taxa'])
    sp_name = a.loc[1, 'Taxa']
    sp = sp_name
    # Assuming you have a way to get the specific dataset for this taxa
    # dataset = get_species_dataset(taxa) 
    ticks.update()

    for j, s in enumerate(S_levels):

        factor = s
        for rep in range(1, replicates + 1):
            #print(f"species {sp_name} s {s} rep {seed}/100")
            # 1. Filter or Prepare the specific dataset slice
            mask = ((a['S'] == s))
            x = a.loc[mask, variables]
            Y = a.loc[mask, 'Occurrence']

            # 2. Run your function
            metrics_df, shap_df =  features_ml_models(x, Y, seed, variables, sp, factor, i, j, is_binary=True, is_balanced=False, plot=False)
            

            # 3. Store Metrics
            # Add metadata columns
            metrics_df['Taxa'] = sp_name
            metrics_df['S'] = s
            metrics_df['PA_N'] = "Weighted"
            metrics_df['Replicate'] = rep
            
            metrics_rows_w.append(metrics_df)
            
            # 4. Store SHAP Values
            # Key: (Taxa, S, Replicate, PA_N) -> Value: (n, 5) DataFrame
            key = (sp_name, s, rep, "Weighted")
            shap_values_store_w[key] = shap_df

            seed += 1

            tocks.update()


# 5. Combine all metrics into one DataFrame
final_metrics_df_w = pd.concat(metrics_rows_w, ignore_index=True)

manager.stop()




Plot the 13 BeeswarmShap (Figure S2)

In [ ]:
import matplotlib.pyplot as plt
import shap

# You already have:
# explanations_list : list of 26 SHAP Explanation objects
# titles_list       : list of 26 title strings

plt.rcParams.update({
    'font.size':        22,
    'axes.titlesize':   26,
    'axes.labelsize':   24,
    'xtick.labelsize':  20,
    'ytick.labelsize':  20,
    'legend.fontsize':  20,
})

n_plots = len(explanations_list)  # should be 26
n_cols = 4
n_rows =  -(-n_plots // n_cols)


fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(24
             , n_rows *8),   # you control size HERE
    squeeze=True
)

idx = 0
for i in range(n_rows):
    for j in range(n_cols):
        ax = axes[i, j]


        if idx < n_plots:
            # Make this axis the active one
            plt.sca(ax)

            # Draw beeswarm INTO this axis
            shap.plots.beeswarm(
                explanations_list[idx],
                plot_size=None,     # REQUIRED when embedding in your own figure
                show=False          # prevents opening separate windows
            )

                    # ── Italic species name ──────────────────────────────────────────
            ax.set_title(titles_list[idx], fontsize=22, fontstyle='italic')
            idx += 1
        else:
            # ── Hide empty axes (no blank frames in the last row) ────────────
            ax.set_visible(False)


plt.tight_layout()
plt.savefig("FigureS2.pdf", dpi=300, bbox_inches="tight")
plt.show()
